## Importing libraries

In [129]:
from bs4 import BeautifulSoup
import requests
import pandas as pd

In [130]:
# def auto_Scrapper_Class(
#     html_tag,
#     course_case,
#     tag_class,
#     attribute='text'
# ):

#     for i in range(1, 50):

#         url = f"https://www.coursera.org/courses?page={i}"

#         page = requests.get(url)
#         soup = BeautifulSoup(page.content, 'html.parser')

#         elements = soup.find_all(
#             html_tag,
#             class_=tag_class
#         )

#         if len(elements) == 0:
#             continue

#         for el in elements:

#             if attribute == 'text':

#                 value = el.get_text(strip=True)

#             else:

#                 value = el.get(attribute)

#             if value:

#                 # khusus href Coursera
#                 if attribute == 'href':
#                     value = "https://www.coursera.org" + value

#                 course_case.append(value)

#             else:

#                 course_case.append(None)

## Define scraping funtion

In [131]:
def auto_Scrapper_Class(
        html_tag,
        course_case,
        tag_class, 
        div_class=None,
        attribute='text'
    ):
    """
    The function auto_Scrapper_Class is used to get three parameters that is the tag,what to scrap and get the content scrapped and class it belongs. 
    """
    for i in range(1,50): # adjust as needed, according to current coursera website, there are 83 pages for all courses
        url = "https://www.coursera.org/courses?page=" +str(i)
        
        #Use below url to gain more customization on the result with different query
        #url = "https://www.coursera.org/search?query=data%20science&page=" +str(i)
        
        page = requests.get(url)
        soup = BeautifulSoup(page.content, 'html.parser')

        if div_class:
            elements = soup.find_all(
                'div',  
                class_ = div_class
            )

            if (len(elements)) != 12:  
                for j in range(0,12):    # There are 12 courses per page
                    course_case.append(None)
                continue

            for name in elements:
                if attribute == 'text':
                    x = name.get_text()
                else: 
                    tag = name.find(html_tag)

                    if tag:
                        x = tag.get(attribute)
                    else:
                        x = None

                if x:
                    if attribute == 'href' and x.startswith('/'):
                        x = "https://www.coursera.org" + x
                    course_case.append(x)
                else:
                    course_case.append(None)        

        else:
            element = soup.find_all(
                html_tag,  
                class_ = tag_class
            )

            if (len(element)) != 12:
                for j in range(0,12):
                    course_case.append(None)
                continue

            for name in element:
                if attribute == 'text':
                    x = name.get_text()
                else:
                    x = name.get(attribute)

                if x:
                    if attribute == 'href' and x.startswith('/'):
                        x = "https://www.coursera.org" + x
                    course_case.append(x)
                else:
                    course_case.append("")


In [132]:
course_title = []
course_organization = []
course_difficulty = []
course_skills = []
course_link = []

In [133]:
#scrap the course title
auto_Scrapper_Class('h3',course_title, tag_class='cds-CommonCard-title css-6ecy9b')

In [134]:
#scrap the other information as per coursera's website html
#auto_Scrapper_Class('div',course_Certificate_type,'_jen3vs _1d8rgfy3')
auto_Scrapper_Class('p',course_difficulty,'cds-119 cds-Typography-base css-dmxkm1 cds-121', 'cds-CommonCard-metadata')
auto_Scrapper_Class('p',course_skills,'cds-119 cds-Typography-base css-dmxkm1 cds-121','cds-CommonCard-bodyContent' )
auto_Scrapper_Class('a', course_link,'cds-119 cds-113 cds-115 cds-CommonCard-titleLink css-fdx774 cds-142',attribute='href')

## Clean the scraped data

In [135]:
data = {
    'Title': course_title,
    'Skills': course_skills,
    'Level': course_difficulty,
    'Link': course_link
}
df = pd.DataFrame(data)
df['Skills'] = df['Skills'].str.replace("Skills you'll gain:", '', regex=False)
df['Level'] = df['Level'].str.split('·').str[0].str.strip()
df

,Title,Skills,Level,Link
0,Google AI,"Vibe coding, Prompt Patterns, AI powered crea...",Beginner,https://www.coursera.org/professional-certific...
1,Google Data Analytics,"Data Storytelling, Rmarkdown, Data Visualizat...",Beginner,https://www.coursera.org/professional-certific...
2,Google Project Management,"Quality Management, Project Closure, Scope Ma...",Beginner,https://www.coursera.org/professional-certific...
3,Google AI Essentials,"Prompt Patterns, Google Gemini, Generative AI...",Beginner,https://www.coursera.org/specializations/ai-es...
4,Google Cybersecurity,"Threat Modeling, Network Security, Threat Man...",Beginner,https://www.coursera.org/professional-certific...
...,...,...,...,...
583,Liberty Mutual Insurance Sales Agent,"Insurance, Insurance Policies, Closing (Sales...",Beginner,https://www.coursera.org/professional-certific...
584,Making Architecture,"Creative Thinking, Creative Problem-Solving, ...",Beginner,https://www.coursera.org/learn/making-architec...
585,"Food Sustainability, Mindful Eating, and Healt...","Food and Beverage, Cooking, Sustainable Devel...",Beginner,https://www.coursera.org/specializations/food-...
586,Social and Multimedia Content Creation,"Video Production, Photo/Video Production and ...",Beginner,https://www.coursera.org/specializations/socia...


In [ ]:
df.to_csv("csv/coursera_course_dataset.csv")

In [137]:
df.isna().sum()

Title     24
Skills    48
Level      0
Link       0
dtype: int64

In [138]:
df['Level'].unique()

array(['Beginner', 'Advanced', 'Intermediate', 'Mixed'], dtype=object)

In [139]:
df[df['Level'] == 'Beginner'].count()

Title     425
Skills    409
Level     441
Link      441
dtype: int64

In [140]:
df[df['Level'] == 'Intermediate'].count()

Title      97
Skills     91
Level     104
Link      104
dtype: int64

In [141]:
df[df['Level'] == 'Advanced'].count()

Title     10
Skills     9
Level     10
Link      10
dtype: int64

In [142]:
df.isna().sum()

Title     24
Skills    48
Level      0
Link       0
dtype: int64

In [143]:
df.head(20)

,Title,Skills,Level,Link
0,Google AI,"Vibe coding, Prompt Patterns, AI powered crea...",Beginner,https://www.coursera.org/professional-certific...
1,Google Data Analytics,"Data Storytelling, Rmarkdown, Data Visualizat...",Beginner,https://www.coursera.org/professional-certific...
2,Google Project Management,"Quality Management, Project Closure, Scope Ma...",Beginner,https://www.coursera.org/professional-certific...
3,Google AI Essentials,"Prompt Patterns, Google Gemini, Generative AI...",Beginner,https://www.coursera.org/specializations/ai-es...
4,Google Cybersecurity,"Threat Modeling, Network Security, Threat Man...",Beginner,https://www.coursera.org/professional-certific...
5,Google IT Support,"IT Security Architecture, Computer Networking...",Beginner,https://www.coursera.org/professional-certific...
6,Google Digital Marketing & E-commerce,"Data Storytelling, Media Planning, Social Med...",Beginner,https://www.coursera.org/professional-certific...
7,Google UX Design,"Responsive Web Design, Storyboarding, Wirefra...",Beginner,https://www.coursera.org/professional-certific...
8,AI For Everyone,"AI Product Strategy, Responsible AI, Data Eth...",Beginner,https://www.coursera.org/learn/ai-for-everyone
9,Machine Learning,"Unsupervised Learning, Supervised Learning, M...",Beginner,https://www.coursera.org/specializations/machi...


## Extras: 
#### In my case, I only want the name of skills that can be obtained from courses in Coursera

In [144]:
skills_column = df['Skills']

skills_column = [str(skill) if skill is not None else '' for skill in skills_column]

# Concatenate all skills into a single string
all_skills_text = ', '.join(skills_column)

# Split the string into a list of skills
all_skills_list = [skill.strip() for skill in all_skills_text.split(',')]

# Get unique skills
distinct_skills = list(set(all_skills_list))

distinct_skills_df = pd.DataFrame({'Distinct Skills': distinct_skills,'source':"coursera"})
len(distinct_skills)

1869

In [145]:
distinct_skills_df

,Distinct Skills,source
0,,coursera
1,Data Governance,coursera
2,Continuous Improvement Process,coursera
3,Medical Practices and Procedures,coursera
4,Cyber Threat Hunting,coursera
...,...,...
1864,Network Troubleshooting,coursera
1865,Java Programming,coursera
1866,Private Equity,coursera
1867,Office Procedures,coursera


In [146]:
#distinct_skills_df.to_csv("skills_coursera.csv")

### Limitation
Unfortunately due to Coursera's website design, we can only get a max of 84 pages(around 1000 courses) with our code. However, there are actually more than 13k courses in Coursera.

Feel free to improve the code <br />
Thank you!